In [ ]:
using SumOfSquares
using DynamicPolynomials
using JuMP
using MultivariatePolynomials
using LinearAlgebra
using Random
using Combinatorics
using SCS
using JSON3
using CSV
using DataFrames
using Dates

const MOI = JuMP.MOI
const DEFAULT_SOLVER = "SCS"


DS helper functions

In [ ]:
function optimizer_factory(solver_kind::AbstractString = DEFAULT_SOLVER)
    normalized = uppercase(strip(solver_kind))
    if normalized == "SCS"
        return optimizer_with_attributes(
            SCS.Optimizer,
            "max_iters" => 200_000,
            "eps_abs" => 1e-5,
            "eps_rel" => 1e-5,
            "eps_infeas" => 1e-12,
            "normalize" => 1,
        )
    elseif normalized == "MOSEK"
        return optimizer_with_attributes(Mosek.Optimizer)
    end
end

function multilinear_monomials(x, d, lowest_deg = 1)
    monoms = Any[]
    for degree in lowest_deg:d
        for combo in combinations(1:length(x), degree)
            push!(monoms, prod(x[idx] for idx in combo))
        end
    end
    return monoms
end

function get_second_derivative(f, variables, i, j)
    return differentiate(differentiate(f, variables[i]), variables[j])
end

function build_multilin_var(model, x, monom, name)
    w = @variable(model, [monom], base_name = name)
    f = sum(w[m] * m for m in monom)
    return f, w
end

function f_sos_submod(model, x, f, t)
    n = length(x)
    for i in 1:n
        for j in i+1:n
            d2f = get_second_derivative(f, x, i, j)
            domain = algebraicset([xi^2 - xi for xi in x if (xi != x[i]) && (xi != x[j])])
            @constraint(model, -d2f >= 0, domain = domain, maxdegree = 2 * t)
        end
    end
end

function get_soln(wg, wh, monoms, x)
    solution_g = Dict()
    solution_h = Dict()

    for monom in monoms
        tup = monom isa typeof(x[1]) ? Tuple([idx for idx in 1:length(x) if x[idx] == monom]) : Tuple([idx for idx in 1:length(x) if x[idx] in monom.vars])
        if abs(value(wg[monom])) > 1e-5
            solution_g[tup] = value(wg[monom])
        end
        if abs(value(wh[monom])) > 1e-5
            solution_h[tup] = value(wh[monom])
        end
    end

    return solution_g, solution_h
end

function trivial_decomp(f_coeffs)
    g = Dict()
    h = Dict()
    for (combo, coeff) in f_coeffs
        if coeff <= 0
            g[combo] = coeff
        else
            h[combo] = -coeff
        end
    end
    return g, h
end

function can_extract_solution(term_status, prim_status)
    return term_status in (MOI.OPTIMAL, MOI.LOCALLY_SOLVED, MOI.SLOW_PROGRESS) && prim_status in (MOI.FEASIBLE_POINT, MOI.NEARLY_FEASIBLE_POINT)
end

function sos_2nd_deriv(f_coeffs, n, d, t; solver_kind::AbstractString = DEFAULT_SOLVER)
    model = SOSModel(optimizer_factory(solver_kind))
    set_silent(model)

    @polyvar x[1:n]
    monoms = multilinear_monomials(x, d)
    g, wg = build_multilin_var(model, x, monoms, "g")
    h, wh = build_multilin_var(model, x, monoms, "h")

    for (combo, coeff) in f_coeffs
        monom = prod(x[idx] for idx in combo)
        @constraint(model, wg[monom] - wh[monom] == coeff)
    end

    f_sos_submod(model, x, g, t)
    f_sos_submod(model, x, h, t)

    weights(r) = r >= 2 ? binomial(r, 2) * 2.0^(n - (r - 2)) : 0.0
    @objective(model, Max, sum(weights(length(combo)) * wh[prod(x[idx] for idx in combo)] for (combo, _) in f_coeffs if 2 <= length(combo) <= d))

    optimize!(model)

    term_status = termination_status(model)
    prim_status = primal_status(model)
    solve_time = try
        MOI.get(backend(model), MOI.SolveTimeSec())
    catch
        NaN
    end

    best_g, best_h = nothing, nothing
    if can_extract_solution(term_status, prim_status)
        best_g, best_h = get_soln(wg, wh, monoms, x)
    end

    return best_g, best_h, solve_time, string(term_status), string(prim_status)
end

function generate_f_coeffs(n, d; min_val = -10, max_val = 10, seed = 0)
    Random.seed!(seed)
    f_coeffs = Dict()
    for degree in 1:d
        for combo in combinations(1:n, degree)
            f_coeffs[Tuple(combo)] = rand(min_val:max_val)
        end
    end
    return f_coeffs
end

function coeffs_to_dataframe(coeffs)
    if coeffs === nothing
        return DataFrame(term = String[], coeff = Float64[])
    end
    rows = sort(collect(coeffs); by = row -> (length(row[1]), collect(row[1])))
    return DataFrame(term = [join(combo, ",") for (combo, _) in rows], coeff = [value for (_, value) in rows])
end


Small decomposition demo.


In [ ]:
demo_n = 5
demo_d = 2
demo_t = 0
demo_seed = 0

f_coeffs = generate_f_coeffs(demo_n, demo_d; seed = demo_seed)
g_triv, h_triv = trivial_decomp(f_coeffs)
g_sos, h_sos, solve_time, status_code, status_text = sos_2nd_deriv(f_coeffs, demo_n, demo_d, demo_t; solver_kind = "SCS")

summary = DataFrame(
    method = ["trivial", "sos"],
    g_terms = [length(g_triv), g_sos === nothing ? missing : length(g_sos)],
    h_terms = [length(h_triv), h_sos === nothing ? missing : length(h_sos)],
    solve_time = [missing, solve_time],
    status = ["n/a", status_code * " / " * status_text],
)

summary


In [ ]:
first(coeffs_to_dataframe(g_sos), 5)
